In [1]:
%pip install ucimlrepo


[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

In [3]:
ds = fetch_ucirepo(id=697)
X, y = ds.data.features, ds.data.targets
df = pd.concat([X, y], axis=1)

In [4]:
print(df.shape)                                # confirma filas y columnas
print(df.dtypes.value_counts())
print(y.iloc[:, 0].value_counts(normalize=True))  # distribución del objetivo
print(df.isna().sum().sum(), df.duplicated().sum())
print(ds.variables[['name', 'role', 'type']])  # lista completa de variables

(4424, 37)
int64      29
float64     7
str         1
Name: count, dtype: int64
Target
Graduate    0.499322
Dropout     0.321203
Enrolled    0.179476
Name: proportion, dtype: float64
0 0
                                              name     role         type
0                                   Marital Status  Feature      Integer
1                                 Application mode  Feature      Integer
2                                Application order  Feature      Integer
3                                           Course  Feature      Integer
4                       Daytime/evening attendance  Feature      Integer
5                           Previous qualification  Feature      Integer
6                   Previous qualification (grade)  Feature   Continuous
7                                      Nacionality  Feature      Integer
8                           Mother's qualification  Feature      Integer
9                           Father's qualification  Feature      Integer
10         

In [5]:
variables = {
    "Género": "Gender",
    "Edad": "Age at enrollment",
    "Nacionalidad": "Nacionality",
    "Beca": "Scholarship holder",
    "Necesidades especiales": "Educational special needs",
    "Internacional": "International",
}
objetivo = y.columns[0]

for etiqueta, columna in variables.items():
    if columna not in df.columns:
        print(f"\n{etiqueta}: no se encontró la columna {columna!r}")
        continue

    grupo = df[columna]
    if etiqueta == "Edad":
        grupo = pd.cut(
            pd.to_numeric(grupo, errors="coerce"),
            bins=[0, 20, 25, 30, 40, float("inf")],
            labels=["≤20", "21–25", "26–30", "31–40", ">40"],
            include_lowest=True,
        )

    resumen = grupo.value_counts(dropna=False).rename_axis("Grupo").to_frame("n")
    resumen["%"] = (100 * resumen["n"] / len(df)).round(1)

    print(f"\n{etiqueta} — tamaño total: {len(df)}")
    print(resumen.to_string())

    tabla = pd.crosstab(
        grupo.astype("string").fillna("Faltante"),
        df[objetivo].astype("string").fillna("Faltante"),
        margins=True,
    )
    print(f"\n{etiqueta} × {objetivo} (recuentos)")
    print(tabla.to_string())


Género — tamaño total: 4424
          n     %
Grupo            
0      2868  64.8
1      1556  35.2

Género × Target (recuentos)
Target  Dropout  Enrolled  Graduate   All
Gender                                   
0           720       487      1661  2868
1           701       307       548  1556
All        1421       794      2209  4424

Edad — tamaño total: 4424
          n     %
Grupo            
≤20    2551  57.7
21–25   828  18.7
31–40   437   9.9
26–30   383   8.7
>40     225   5.1

Edad × Target (recuentos)
Target             Dropout  Enrolled  Graduate   All
Age at enrollment                                   
21–25                  295       182       351   828
26–30                  229        57        97   383
31–40                  241        58       138   437
>40                    114        26        85   225
≤20                    542       471      1538  2551
All                   1421       794      2209  4424

Nacionalidad — tamaño total: 4424
          n     %
Gru

In [6]:
pd.crosstab(df['Curricular units 2nd sem (enrolled)'] == 0, df['Target'], normalize='index')
pd.crosstab(df['Curricular units 1st sem (enrolled)'] == 0, df['Target'], normalize='index')

Target,Dropout,Enrolled,Graduate
Curricular units 1st sem (enrolled),,,
False,0.316682,0.180490,0.502828
True,0.427778,0.155556,0.416667


In [7]:
for s in ['1st', '2nd']:
    col = f'Curricular units {s} sem (enrolled)'
    print(s, pd.crosstab(df[col] == 0, df['Target']))             # recuentos
    print(s, pd.crosstab(df[col] == 0, df['Target'], normalize='index').round(3))

# ¿los de 0 matriculadas en el 1.er semestre tienen unidades acreditadas?
z = df[df['Curricular units 1st sem (enrolled)'] == 0]
print(z['Curricular units 1st sem (credited)'].describe())

1st Target                               Dropout  Enrolled  Graduate
Curricular units 1st sem (enrolled)                             
False                                   1344       766      2134
True                                      77        28        75
1st Target                               Dropout  Enrolled  Graduate
Curricular units 1st sem (enrolled)                             
False                                  0.317     0.180     0.503
True                                   0.428     0.156     0.417
2nd Target                               Dropout  Enrolled  Graduate
Curricular units 2nd sem (enrolled)                             
False                                   1344       766      2134
True                                      77        28        75
2nd Target                               Dropout  Enrolled  Graduate
Curricular units 2nd sem (enrolled)                             
False                                  0.317     0.180     0.503
True     

In [8]:
%pip install scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

y = (df['Target'] == 'Dropout').astype(int)
cv = StratifiedKFold(5, shuffle=True, random_state=42)
res = {}
for c in df.columns.drop('Target'):
    t = DecisionTreeClassifier(max_depth=4, min_samples_leaf=30, random_state=42)
    res[c] = cross_val_score(t, df[[c]], y, cv=cv, scoring='roc_auc').mean()
print(pd.Series(res).sort_values(ascending=False).round(3))

Curricular units 2nd sem (approved)               0.861
Curricular units 1st sem (approved)               0.822
Curricular units 2nd sem (grade)                  0.812
Curricular units 1st sem (grade)                  0.775
Curricular units 2nd sem (evaluations)            0.698
Curricular units 1st sem (evaluations)            0.677
Age at enrollment                                 0.672
Application mode                                  0.660
Tuition fees up to date                           0.649
Previous qualification (grade)                    0.643
Course                                            0.640
Scholarship holder                                0.614
Curricular units 1st sem (enrolled)               0.612
Curricular units 2nd sem (enrolled)               0.609
Gender                                            0.604
Admission grade                                   0.581
Father's qualification                            0.581
Debtor                                          

In [10]:
c1 = 'Curricular units 1st sem (enrolled)'
c2 = 'Curricular units 2nd sem (enrolled)'
print(((df[c1] == 0) == (df[c2] == 0)).all())
z = df[df[c1] == 0]
print(z.filter(like='Curricular units').describe().T[['min', 'max']])
print(z['Course'].value_counts().head(10))
print(z['Application mode'].value_counts().head(5))

True
                                                min  max
Curricular units 1st sem (credited)             0.0  0.0
Curricular units 1st sem (enrolled)             0.0  0.0
Curricular units 1st sem (evaluations)          0.0  0.0
Curricular units 1st sem (approved)             0.0  0.0
Curricular units 1st sem (grade)                0.0  0.0
Curricular units 1st sem (without evaluations)  0.0  0.0
Curricular units 2nd sem (credited)             0.0  0.0
Curricular units 2nd sem (enrolled)             0.0  0.0
Curricular units 2nd sem (evaluations)          0.0  0.0
Curricular units 2nd sem (approved)             0.0  0.0
Curricular units 2nd sem (grade)                0.0  0.0
Curricular units 2nd sem (without evaluations)  0.0  0.0
Course
171    180
Name: count, dtype: int64
Application mode
1     115
17     29
43     11
39      9
44      6
Name: count, dtype: int64


In [11]:
print(df['Course'].value_counts().get(171))
print(pd.crosstab(df['Course'] == 171, df['Curricular units 1st sem (enrolled)'] == 0))

215
Curricular units 1st sem (enrolled)  False  True 
Course                                           
False                                 4209      0
True                                    35    180


In [12]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

y = (df['Target'] == 'Dropout').astype(int)
cv = StratifiedKFold(5, shuffle=True, random_state=42)
res = {}
for c in df.columns.drop('Target'):
    t = DecisionTreeClassifier(max_depth=4, min_samples_leaf=30, random_state=42)
    res[c] = cross_val_score(t, df[[c]], y, cv=cv, scoring='roc_auc').mean()
print(pd.Series(res).sort_values(ascending=False).round(3))

Curricular units 2nd sem (approved)               0.861
Curricular units 1st sem (approved)               0.822
Curricular units 2nd sem (grade)                  0.812
Curricular units 1st sem (grade)                  0.775
Curricular units 2nd sem (evaluations)            0.698
Curricular units 1st sem (evaluations)            0.677
Age at enrollment                                 0.672
Application mode                                  0.660
Tuition fees up to date                           0.649
Previous qualification (grade)                    0.643
Course                                            0.640
Scholarship holder                                0.614
Curricular units 1st sem (enrolled)               0.612
Curricular units 2nd sem (enrolled)               0.609
Gender                                            0.604
Admission grade                                   0.581
Father's qualification                            0.581
Debtor                                          

In [14]:

df.columns = df.columns.str.strip()          # algunas versiones traen tabulaciones en los nombres
df["dropout"] = (df["Target"] == "Dropout").astype(int)

# 1) Tamaños de subgrupo y tasa de abandono en cada uno
for col in ["Gender", "Scholarship holder", "Debtor", "Tuition fees up to date",
            "Daytime/evening attendance", "Displaced", "International",
            "Educational special needs"]:
    print(df.groupby(col)["dropout"].agg(n="size", tasa="mean").round(3), "\n")

# 2) Macroeconómicas: ¿actúan como identificador de cohorte?
macro = ["Unemployment rate", "Inflation rate", "GDP"]
g = df.groupby(macro)["dropout"].agg(n="size", tasa="mean").round(3)
print(len(g), "combinaciones distintas"); print(g)

           n   tasa
Gender             
0       2868  0.251
1       1556  0.451 

                       n   tasa
Scholarship holder             
0                   3325  0.387
1                   1099  0.122 

           n   tasa
Debtor             
0       3921  0.283
1        503  0.620 

                            n   tasa
Tuition fees up to date             
0                         528  0.866
1                        3896  0.247 

                               n   tasa
Daytime/evening attendance             
0                            483  0.429
1                           3941  0.308 

              n   tasa
Displaced             
0          1998  0.376
1          2426  0.276 

                  n   tasa
International             
0              4314  0.322
1               110  0.291 

                              n   tasa
Educational special needs             
0                          4373  0.321
1                            51  0.333 

10 combinaciones distintas
     